<a href="https://colab.research.google.com/github/frankausberlin/deen-lupysta/blob/main/scripts/dsdash.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@markdown <font size='+2' color='#005F6A'>**OpenRouterModels**</font><br>
#@markdown * Uses **openrouter-cli**-pip to get model infos from OpenRouter. It **must be installed**: `uv tool install openrouter-cli` or `pip install openrouter-cli`
#@markdown * You must insert your **OpenRouter API-key** first with: `openrouter-cli configure`'.
#@markdown * You can **specify** a comma-separated **list of frontier models** to be **scanned**, whose **latest versions** will be displayed in the **overview**.
from ipywidgets import Button, Box, Layout, Textarea, VBox, HTML, HBox
import subprocess, json, pprint

class ORMButtonBox ():
  def __init__ (self, descriptions, clicker=None, maxchar=60, color=None):
    # remember stuff - list to set
    self.descriptions = descriptions if len (descriptions) == len (set (descriptions)) else set (descriptions)
    self.clicker, self.maxchar, self.color, self.position, self.button = clicker, maxchar, color, -1, None
    self._original_colors = {} # Dictionary to store original colors

    # make buttons
    self.buttons = [Button (description = i if len (i) <= maxchar else f'{i[:(maxchar-3)]}...',
                            layout      = Layout(width='auto', height='22px'),
                            tooltip     = f'{i}')
                    for i in descriptions]

    # put them in a box / bind event
    self.widget = Box (layout   = Layout (display='flex', flex_flow='wrap'),
                       children = self.buttons)
    for button in self.buttons:
      self._original_colors[button] = button.style.button_color # Store original color
      button.on_click (self._clicker)

  # An alternative for the Tab widget
  def _clicker (self, b):
    # search clicked button and remember
    self.position, self.button = [(i,but) for i, but in enumerate(self.buttons) if b == but][0]

    # unselect (color) all buttons and select new (list of colors here: https://www.quackit.com/css/css_color_codes.cfm)
    self.unselect()
    if self.color: self.buttons[self.position].style={'button_color':self.color}
    self.buttons[self.position].layout.border = '1px solid black'

    # fire event
    if self.clicker: self.clicker (self)

  def unselect (self):
    # unselect all buttons and restore original color
    for b in self.buttons:
      b.layout.border = ''
      b.style = {'button_color': self._original_colors.get(b, None)} # Restore original color, default to None if not found

def selectOrganization (bbox):
  global selectedOrga
  selectedOrga =  bbox.button.tooltip
  org_models = [m for m in json_list if m['id'].split('/')[0] == selectedOrga]
  modelsBox = ORMButtonBox ([i['id'].replace (bbox.button.tooltip+'/','') for i in org_models], clicker=selectModel, color='powderblue')
  widgetList.children=(widgetList.children[0], widgetList.children[1], modelsBox.widget)

def selectModel (bbox):
  for model in json_list:
    if model['id'] == selectedOrga+'/'+bbox.button.tooltip:
      if len (widgetList.children) < 4:
        widgetList.children = (*widgetList.children, Textarea (layout=Layout(width='auto', height='500px')))
      widgetList.children[3].value = pprint.pformat(model, width=160)

# ___________________________________________________________
#|______________________hello_component______________________|
try:
  json_list = json.loads(subprocess.run(['openrouter-cli','models', '--raw'], capture_output=True, text=True, check=True).stdout)
  tab_list = subprocess.run(['openrouter-cli','models'], capture_output=True, text=True, check=True).stdout.split('\n')
except: json_list = []

if json_list:

  # generate frontier models overview
  frontier_models = "gpt-5, claude-opus-4, claude-sonnet-4, gemini-3, deepseek-v4, minimax-m2, kimi-k2, glm-5, qwen3, mimo-v2, grok-4"  #@param {type:"string"}
  ignore_frontier_models_with = "8b, 27b, 35b, a3b, -lite, -image, -micro, -nano, qwen3.6-flash"  #@param {type:"string"}
  ignore = ignore_frontier_models_with.replace(' ', '').split(',')

  # build the list with tuples ('versionnr', 'model-id') for the frontier models
  frontiers = frontier_models.replace(' ', '').split(',')

  # correct filter
  bad_id = lambda id: sum([f in id for f in ignore])

  frontier_version_id_list = [(j['id'].split('/')[-1].split(f[:-1])[-1].split('-')[0] , j['id'])
                              for f in frontiers for j in json_list
                                if f in j['id'] and
                                  ':free' not in j['id'] and
                                  j['id'].split('/')[-1].split(f[:-1])[-1].split('-')[0][-1] in '0123456789' and
                                  not bad_id (j['id'])]

  # build the frontier model overview
  tmp = []
  for frontier in frontiers:
    match = sorted ([f for f in frontier_version_id_list if frontier in f[1]])
    if not match: continue # catch empty lists due to rigorous filtering

    show_version = match[-1][0]
    versions = [f for f in match if show_version == f[0]]
    hm = f"<font color=blue size=3><b>{versions[0][1].split('/')[0]}</b></font><hr>"
    for version in versions:
      hm += f"<b>{version[1].split('/')[1]}</b><br>"
      idj = [j for j in json_list if j['id'] == version[1]][0]
      r, w  = f"{(float (idj['pricing']['prompt'])*1000000):.2f}", f"{(float (idj['pricing']['completion'])*1000000):.2f}"
      cr = f"{(float(idj['pricing']['input_cache_read'])*1000000):.2f}" if 'input_cache_read' in idj['pricing'] else ' '
      cw = f"{(float(idj['pricing']['input_cache_write'])*1000000):.2f}" if 'input_cache_write' in idj['pricing'] else ' '
      hm += f"Context: {idj['context_length']}<br>Price: {r} / {w}<br>[Cache: {cr} / {cw}]<br>"

    tmp.append (HTML(value=hm[:-4], layout={"border":"2px solid yellow"}))
  overview_panel = HBox (children=tmp)

  # build orga / model selector
  selectedOrga      = None
  organization_list = sorted (set ([j['id'].split('/')[0] for j in json_list if '/' in j['id']]))
  organizationsBox  = ORMButtonBox (organization_list, clicker=selectOrganization, color='powderblue')
  title             = HTML (value="<font color=green size=4><b>Overview Frontier Models Latest")
  widgetList        = VBox (children=[organizationsBox.widget,HTML(value='<hr>')])

  display (VBox (children=[title,overview_panel]), widgetList)

In [ ]:
#@markdown <font size='+2' color='#005F6A'>**Hosting: CudaOnHostTest**</font><br>Check if cuda runing on host and is available for **pytorch, tensorflow, jax** and **numba**.

import os, warnings

# if you are using > Python 3.11:
# with warnings.catch_warnings(action="ignore"):

with warnings.catch_warnings():
  warnings.simplefilter("ignore")

  try:
    import torch
    cuda_pytorch = torch.cuda.is_available()
  except: cuda_pytorch = False

  try:
    import tensorflow as tf
    cuda_tensorflow = ':GPU:' in str(tf.config.list_physical_devices('GPU'))
  except: cuda_tensorflow = False

  try:
    import jax
    cuda_jax = len(jax.devices()) > 0
  except: cuda_jax = False

  try:
    from numba import cuda
    cuda_numba = cuda.is_available()
  except: cuda_numba = False

  try:
    from IPython.display import clear_output
    clear_output ()
  except: pass


  hasNvidiaSmi  = os.system ('nvidia-smi --version >/dev/null 2>&1') == 0
  if hasNvidiaSmi:
    print (f"pytorch-cuda:     \x1b[92mTrue\x1b[0m") if cuda_pytorch     else print (f"pytorch-cuda:     \x1b[91mFalse\x1b[0m")
    print (f"tensorflow-cuda:  \x1b[92mTrue\x1b[0m") if cuda_tensorflow  else print (f"tensorflow-cuda:  \x1b[91mFalse\x1b[0m")
    print (f"jax-cuda:         \x1b[92mTrue\x1b[0m") if cuda_jax         else print (f"jax-cuda:         \x1b[91mFalse\x1b[0m")
    print (f"numba-cuda:       \x1b[92mTrue\x1b[0m\n") if cuda_numba       else print (f"numba-cuda:       \x1b[91mFalse\x1b[0m")
    !nvidia-smi

  else:
    print ('\x1b[91mno cuda on host\x1b[0m')


In [ ]:
#@markdown <table><tr><td><img src='https://drive.google.com/uc?id=1gzb7d4L1Ww_uEwCh3N5mJfhRFPJxJSpy' width='250' alt='alt text'></td><td width='800'>
#@markdown <table><tr><td align='center' width='800'><font size='+5'><b>Matrix multiplication</b><br><br></font></td></tr></table><font size='+1'>
#@markdown <li> This snippet contains different versions of matrix multiplication using the mnist digit dataset.
#@markdown <li> In Python, you can calculate $a \cdot b$ in several ways, the short one is <code>a @ b</code>.
#@markdown <li> You can watch a <a href='https://www.youtube.com/watch?v=Tf-8F5q8Xww'>video</a> by Jeremy Howard - lesson 11 of 'Deeplearning for Coders', 59:15
#@markdown <li> The different implementations of matrix multiplication illustrate very well some basic concepts of Python.
#@markdown <br><br><br><br><br><br><br><br></td></tr></table>

# v11 dlc - 59:15 - Matrix multiplication from scratch


import time
# --- 0. Start Global Timer ---
start_time = time.time()

from torchvision  import datasets
from numpy        import array, zeros as zerosA, ndarray
from torch        import tensor, zeros as zerosT, dot as dot_torch, einsum, Tensor
from numba        import njit
from random       import seed, random
from typing       import List, Tuple
# --- JAX Imports ---
import jax
import jax.numpy as jnp
from jax import jit

# --- 1. Data Preparation ---
seed(1)

mnistTrain        = datasets.MNIST ('/tmp/mnistdata', train=True, download=True)
mnist_sample      = [list(mnistTrain[i][0].getdata()) for i in range (5)]
mnist_complete    = [list(mnistTrain[i][0].getdata()) for i in range (len(mnistTrain))]
weights           = [[random() for _ in range(10)] for all in range(784)]
bias              = [0.0 for _ in range(10)]

# Prepare JAX specific arrays (DeviceArrays, automatically pushed to GPU if available)
mnist_jax         = jnp.array(mnist_complete, dtype=jnp.float32)
weights_jax       = jnp.array(weights, dtype=jnp.float32)

# --- Timing 1: After Data Preparation ---
time_after_data = time.time()
print(f"--- Timing 1: Data preparation took {time_after_data - start_time:.2f} seconds ---\n")


# --- 2. Function Definitions ---

# Pure Python implementation using nested loops.
# Very slow because Python interprets every loop iteration. Only suitable for small subsets.
def matmul_list(a: List[List[float]], b: List[List[float]]) -> List[List[float]]:
  ar: int = len(a)
  ac: int = len(a[0])
  br: int = len(b)
  bc: int = len(b[0])
  t: List[List[float]] = [[0.0 for j in range(bc)] for i in range(ar)]
  for i in range(ar):
    for j in range(bc):
      for k in range(ac):
        t[i][j] += a[i][k] * b[k][j]
  return t

# NumPy implementation but iterating with a custom pure Python dot-product function.
# Still quite slow because the loops are not fully vectorized in C/C++.
def matmul_numpy(a: ndarray, b: ndarray) -> ndarray:
  def _dot(h: ndarray, i: ndarray) -> float:
    res: float = 0.
    for j in range(len(h)): res+=h[j]*i[j]
    return res
  (ar,ac), (br,bc) = a.shape, b.shape
  c: ndarray = zerosA((ar, bc))
  for i in range(ar):
    for j in range(bc): c[i,j] = _dot(a[i,:], b[:,j])
  return c

# Uses Numba's @njit decorator.
# This compiles the custom Python loop into highly optimized machine code (LLVM) just-in-time, making it drastically faster.
def matmul_numba(a: ndarray, b: ndarray) -> ndarray:
  @njit
  def _dot_numba(h: ndarray, i: ndarray) -> float:
    res: float = 0.
    for j in range(len(h)): res+=h[j]*i[j]
    return res

  (ar,ac), (br,bc) = a.shape, b.shape
  c: ndarray = zerosA((ar, bc))
  for i in range(ar):
    for j in range(bc): c[i,j] = _dot_numba(a[i,:], b[:,j])
  return c

# NumPy implementation using row/col indexing and the .sum() method.
# Avoids the innermost Python loop, but explicit indexing in Python loops is still a bottleneck.
def matmul_index_numpy(a: ndarray, b: ndarray) -> ndarray:
  (ar,ac),(br,bc) = a.shape,b.shape
  c: ndarray = zerosA((ar, bc))
  for i in range(ar):
    for j in range(bc): c[i,j] = (a[i,:] * b[:,j]).sum()
  return c

# PyTorch equivalent of the indexed approach.
# Similar performance bottleneck as NumPy because of the explicit Python loop structure.
def matmul_index_torch(a: Tensor, b: Tensor) -> Tensor:
  (ar,ac),(br,bc) = a.shape,b.shape
  c: Tensor = zerosT((ar, bc))
  for i in range(ar):
    for j in range(bc): c[i,j] = (a[i,:] * b[:,j]).sum()
  return c

# NumPy implementation leveraging broadcasting.
# Eliminates one loop by performing operations on entire rows/columns simultaneously. Much faster.
def matmul_broadcast_numpy(a: ndarray, b: ndarray) -> ndarray:
  (ar,ac),(br,bc) = a.shape,b.shape
  c: ndarray = zerosA((ar, bc))
  for i in range(ar):
    c[i] = (a[i,:,None] * b).sum(axis=0)
  return c

# PyTorch equivalent leveraging broadcasting.
def matmul_broadcast_torch(a: Tensor, b: Tensor) -> Tensor:
  (ar,ac),(br,bc) = a.shape,b.shape
  c: Tensor = zerosT((ar, bc))
  for i in range(ar):
    c[i] = (a[i,:,None] * b).sum(dim=0)
  return c

# Highly optimized standard NumPy matrix multiplication operator.
# Offloads the entire calculation to optimized C/C++ backend libraries (like BLAS). Very fast.
def matmul_operator_numpy(a: ndarray, b: ndarray) -> ndarray:
  return a@b

# Highly optimized standard PyTorch matrix multiplication operator running on CPU.
def matmul_operator_torch(a: Tensor, b: Tensor) -> Tensor:
  return a@b

# PyTorch using Einstein summation convention.
# Clean, readable syntax for tensor contractions, heavily optimized under the hood.
def matmul_einstein_torch(a: Tensor, b: Tensor) -> Tensor:
  return einsum('ik,kj->ij', a, b)

# PyTorch matrix multiplication explicitly executed on the GPU.
# Massively parallelized operation using CUDA cores. The fastest option for huge datasets.
def matmul_cuda_tensor(a: Tensor, b: Tensor) -> Tensor:
  return a@b

# Standard JAX matrix multiplication.
# Similar API to NumPy, but naturally prepared to run on accelerators (GPU/TPU) if available.
def matmul_operator_jax(a, b):
    return a @ b

# JAX matrix multiplication wrapped in @jit (Just-In-Time compilation).
# Compiles the operations via XLA for maximum performance. Slow on the first call (compilation overhead), blazingly fast afterwards.
@jit
def matmul_jit_jax(a, b):
    return a @ b

# JIT-compiled Einstein summation in JAX.
@jit
def matmul_einsum_jax(a, b):
    return jnp.einsum('ik,kj->ij', a, b)


# --- Timing 2: After Function Definitions ---
time_after_defs = time.time()
print(f"--- Timing 2: Function definitions took {time_after_defs - time_after_data:.2f} seconds ---\n")


# --- 3. Executions and Benchmarks ---

print('1. matmul_list (with only 10% data)')
%time result = matmul_list (mnist_complete[:6000], weights)

print('\n2. matmul_numpy (with only 10% data)')
%time result = matmul_numpy (array(mnist_complete[:6000]), array(weights))

print('\n3. matmul_numba (with 100% data)')
%time result = matmul_numba (array(mnist_complete), array(weights))

print('\n4. matmul_index_numpy (with 100% data)')
%time result = matmul_index_numpy (array(mnist_complete), array(weights))

print('\n5. matmul_index_torch (with 100% data)')
%time result = matmul_index_torch (tensor(mnist_complete,dtype=float), tensor(weights,dtype=float))

print('\n6. matmul_broadcast_numpy (with 100% data)')
%time result = matmul_broadcast_numpy (array(mnist_complete), array(weights))

print('\n7. matmul_broadcast_torch (with 100% data)')
%time result = matmul_broadcast_torch (tensor(mnist_complete,dtype=float), tensor(weights,dtype=float))

print('\n8. matmul_operator_numpy (with 100% data)')
%time result = matmul_operator_numpy (array(mnist_complete), array(weights))

print('\n9. matmul_operator_torch (with 100% data)')
%time result = matmul_operator_torch (tensor(mnist_complete,dtype=float), tensor(weights,dtype=float))

print('\n10. matmul_einsum_torch (with 100% data)')
%time result = matmul_einstein_torch (tensor(mnist_complete,dtype=float), tensor(weights,dtype=float))

print('\n11. matmul_cuda_torch (with 100% data)')
# Creating CUDA tensors directly inline for the benchmark
%time result = matmul_cuda_tensor (tensor(mnist_complete,dtype=float).cuda(), tensor(weights,dtype=float).cuda())

print('\n12. matmul_operator_jax (with 100% data)')
# .block_until_ready() forces JAX to wait for asynchronous GPU execution to finish so %time measures accurately.
%time result = matmul_operator_jax(mnist_jax, weights_jax).block_until_ready()

print('\n13. matmul_jit_jax (with 100% data) - FIRST CALL (includes XLA compilation)')
%time result = matmul_jit_jax(mnist_jax, weights_jax).block_until_ready()

print('\n14. matmul_jit_jax (with 100% data) - SECOND CALL (pure execution time)')
%time result = matmul_jit_jax(mnist_jax, weights_jax).block_until_ready()

print('\n15. matmul_einsum_jax (with 100% data) - JIT COMPILED (second call)')
_ = matmul_einsum_jax(mnist_jax, weights_jax).block_until_ready() # Warm-up
%time result = matmul_einsum_jax(mnist_jax, weights_jax).block_until_ready()


# --- Timing 3: Total Execution Time ---
time_end = time.time()
print(f"\n--- Timing 3: Total execution time (Preparation + Definitions + Benchmarks) took {time_end - start_time:.2f} seconds ---")